In [ ]:
# Cell 1: Install correct PyTorch for P100, restart once if needed
import subprocess, sys, os
try:
    import torch
    _ok = '2.3.1' in torch.__version__ and torch.cuda.is_available()
except Exception:
    _ok = False

if not _ok:
    print('Installing torch 2.3.1+cu118...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'torch==2.3.1', 'torchaudio==2.3.1',
                    '--index-url', 'https://download.pytorch.org/whl/cu118'], check=True)
    print('Done. Restarting kernel — Run All one more time.')
    os.kill(os.getpid(), 9)
else:
    print(f'torch {torch.__version__} | CUDA OK — proceeding')

In [ ]:
# Cell 2: Everything — config, model, dataset, train, predict
import os, math, random, re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from torch.optim.swa_utils import AveragedModel, update_bn
from tqdm import tqdm

# ── Paths ─────────────────────────────────────────────────────────────────────
def _find_data_dir(base='/kaggle/input'):
    for root, dirs, files in os.walk(base):
        if 'train.csv' in files:
            return root
    raise FileNotFoundError(f'train.csv not found under {base}')

DATA_DIR   = _find_data_dir()
OUTPUT_DIR = '/kaggle/working'
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP    = DEVICE.type == 'cuda'
print('Device:', DEVICE, '| DATA_DIR:', DATA_DIR)

# ── Hyperparams ───────────────────────────────────────────────────────────────
SAMPLE_RATE  = 16_000
N_MELS       = 80
N_FFT        = 512
HOP_LENGTH   = 160
WIN_LENGTH   = 400
F_MIN, F_MAX = 80.0, 7600.0

MODEL_DIM    = 144
NUM_HEADS    = 4
NUM_LAYERS   = 10    # increased from 8 (~4.97M params)
FF_DIM       = MODEL_DIM * 4
CONV_KERNEL  = 31
DROPOUT      = 0.15
LAYER_DROP   = 0.1

BATCH_SIZE   = 16
ACCUM_STEPS  = 2      # effective batch = 32
LR           = 3e-4
WARMUP_STEPS = 4_000
MAX_EPOCHS   = 90
SWA_START    = 55     # 35 epochs of SWA averaging
GRAD_CLIP    = 5.0
FREQ_MASK_P  = 27
TIME_MASK_P  = 70
MAX_MEL_LEN  = 400

# TTA: 3 speeds at inference, majority vote
TTA_SPEEDS  = [0.95, 1.0, 1.05]

# ── Vocabulary ────────────────────────────────────────────────────────────────
BLANK_ID = 0
VOCAB    = ['<blank>', ' '] + list('абвгдеёжзийклмнопрстуфхцчшщъыьэюя')
CHAR2ID  = {c: i for i, c in enumerate(VOCAB)}
ID2CHAR  = {i: c for c, i in CHAR2ID.items()}
VOCAB_SIZE = len(VOCAB)

def encode(text): return [CHAR2ID[c] for c in text.lower() if c in CHAR2ID]
def decode(ids):  return ''.join(ID2CHAR.get(i, '') for i in ids)

# ── Russian number <-> text ───────────────────────────────────────────────────
_ONES_M = ['','один','два','три','четыре','пять','шесть','семь','восемь','девять']
_ONES_F = ['','одна','две','три','четыре','пять','шесть','семь','восемь','девять']
_TEENS  = ['десять','одиннадцать','двенадцать','тринадцать','четырнадцать',
           'пятнадцать','шестнадцать','семнадцать','восемнадцать','девятнадцать']
_TENS   = ['','','двадцать','тридцать','сорок','пятьдесят','шестьдесят','семьдесят','восемьдесят','девяносто']
_HUNDS  = ['','сто','двести','триста','четыреста','пятьсот','шестьсот','семьсот','восемьсот','девятьсот']

def _chunk(n, fem=False):
    parts, h, r = [], n//100, n%100
    if h: parts.append(_HUNDS[h])
    if 10<=r<=19: parts.append(_TEENS[r-10])
    else:
        t, u = r//10, r%10
        if t: parts.append(_TENS[t])
        if u: parts.append(_ONES_F[u] if fem else _ONES_M[u])
    return ' '.join(p for p in parts if p)

def _thou_sfx(n):
    l2, l1 = n%100, n%10
    if 11<=l2<=19: return 'тысяч'
    if l1==1: return 'тысяча'
    if 2<=l1<=4: return 'тысячи'
    return 'тысяч'

def number_to_text(n):
    thou, rem = n//1000, n%1000
    parts = [_chunk(thou, fem=True), _thou_sfx(thou)]
    if rem: parts.append(_chunk(rem, fem=False))
    return ' '.join(p for p in parts if p)

_W2N = {}
_TENS_V  = [0,0,20,30,40,50,60,70,80,90]
_HUNDS_V = [0,100,200,300,400,500,600,700,800,900]
for _i in range(1,10): _W2N[_ONES_M[_i]]=_i; _W2N[_ONES_F[_i]]=_i
for _i,_w in enumerate(_TEENS,10): _W2N[_w]=_i
for _i in range(2,10): _W2N[_TENS[_i]]=_TENS_V[_i]
for _i in range(1,10): _W2N[_HUNDS[_i]]=_HUNDS_V[_i]
_THOU_WORDS = {'тысяча', 'тысячи', 'тысяч'}

def text_to_number(text):
    toks = re.sub(r'[^а-яё ]','',text.lower()).split()
    idx  = next((i for i,t in enumerate(toks) if t in _THOU_WORDS), None)
    if idx is None: return sum(_W2N.get(t,0) for t in toks)
    thou = sum(_W2N.get(t,0) for t in toks[:idx]) or 1
    rem  = sum(_W2N.get(t,0) for t in toks[idx+1:])
    return thou*1000 + rem

def safe_parse(text):
    """text -> int in [1000, 999999], with ×1000 heuristic for missing thousands."""
    has_thou = any(w in text for w in _THOU_WORDS)
    val = text_to_number(text)
    if not has_thou:
        if 1 <= val <= 999:
            val = val * 1000
        elif val <= 0 or val > 999_999:
            return 1_000
    if val <= 0 or val > 999_999:
        return 1_000
    return int(val)

assert number_to_text(280520) == 'двести восемьдесят тысяч пятьсот двадцать'
assert text_to_number('двести восемьдесят тысяч пятьсот двадцать') == 280520
assert safe_parse('пять') == 5000
assert safe_parse('пять тысяч') == 5000
print('number<->text OK')

# ── CTC Greedy decode ─────────────────────────────────────────────────────────
def ctc_greedy_decode(log_probs, lengths=None):
    T, B, _ = log_probs.shape
    best = log_probs.argmax(-1)
    results = []
    for b in range(B):
        T_b = int(lengths[b]) if lengths is not None else T
        dec, prev = [], -1
        for tok in best[:T_b, b].tolist():
            if tok != prev:
                if tok != BLANK_ID: dec.append(tok)
                prev = tok
        results.append(dec)
    return results

# ── Model ─────────────────────────────────────────────────────────────────────
class PositionalEncoding(nn.Module):
    def __init__(self, dim, dropout, max_len=5000):
        super().__init__()
        self.drop = nn.Dropout(dropout)
        pe = torch.zeros(max_len, dim)
        pos = torch.arange(max_len).unsqueeze(1)
        div = torch.exp(torch.arange(0,dim,2)*(-math.log(10000.)/dim))
        pe[:,0::2] = torch.sin(pos*div); pe[:,1::2] = torch.cos(pos*div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x): return self.drop(x + self.pe[:,:x.size(1)])

class ConvSubsampling(nn.Module):
    def __init__(self, n_mels, model_dim):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1,32,3,stride=2,padding=1), nn.ReLU(),
            nn.Conv2d(32,32,3,stride=2,padding=1), nn.ReLU())
        freq_out = math.ceil(math.ceil(n_mels/2)/2)
        self.proj = nn.Linear(32*freq_out, model_dim)
    def _sub(self, l):
        l = torch.div(l+1,2,rounding_mode='floor')
        return torch.div(l+1,2,rounding_mode='floor')
    def forward(self, x, lengths):
        B,T,F = x.shape
        x = self.conv(x.unsqueeze(1))
        _,C,T2,F2 = x.shape
        x = x.permute(0,2,1,3).contiguous().view(B,T2,C*F2)
        return self.proj(x), self._sub(lengths)

class FeedForward(nn.Module):
    def __init__(self, dim, ff_dim, drop):
        super().__init__()
        self.norm=nn.LayerNorm(dim); self.fc1=nn.Linear(dim,ff_dim)
        self.fc2=nn.Linear(ff_dim,dim); self.drop=nn.Dropout(drop)
    def forward(self, x):
        return self.drop(self.fc2(self.drop(F.silu(self.fc1(self.norm(x))))))

class ConvModule(nn.Module):
    def __init__(self, dim, ks, drop):
        super().__init__()
        self.norm=nn.LayerNorm(dim); self.pw1=nn.Conv1d(dim,2*dim,1)
        self.dw=nn.Conv1d(dim,dim,ks,padding=ks//2,groups=dim)
        self.bn=nn.BatchNorm1d(dim); self.pw2=nn.Conv1d(dim,dim,1)
        self.drop=nn.Dropout(drop)
    def forward(self, x):
        r = self.norm(x).transpose(1,2)
        r = F.glu(self.pw1(r),dim=1)
        r = F.silu(self.bn(self.dw(r)))
        return self.drop(self.pw2(r)).transpose(1,2)

class ConformerBlock(nn.Module):
    def __init__(self, dim, heads, ff_dim, ks, drop, layer_drop=0.0):
        super().__init__()
        self.ff1=FeedForward(dim,ff_dim,drop); self.an=nn.LayerNorm(dim)
        self.attn=nn.MultiheadAttention(dim,heads,dropout=drop,batch_first=True)
        self.ad=nn.Dropout(drop); self.conv=ConvModule(dim,ks,drop)
        self.ff2=FeedForward(dim,ff_dim,drop); self.fn=nn.LayerNorm(dim)
        self.layer_drop = layer_drop
    def forward(self, x, mask=None):
        if self.training and self.layer_drop > 0 and random.random() < self.layer_drop:
            return x
        x = x + 0.5*self.ff1(x)
        n = self.an(x); a,_=self.attn(n,n,n,key_padding_mask=mask)
        x = x + self.ad(a); x = x + self.conv(x)
        return self.fn(x + 0.5*self.ff2(x))

class ConformerCTC(nn.Module):
    def __init__(self, vocab_size, n_mels=80, model_dim=144, num_heads=4,
                 num_layers=10, ff_dim=576, conv_kernel=31, dropout=0.1, layer_drop=0.1):
        super().__init__()
        self.sub=ConvSubsampling(n_mels,model_dim)
        self.pos=PositionalEncoding(model_dim,dropout)
        self.blocks=nn.ModuleList([
            ConformerBlock(model_dim,num_heads,ff_dim,conv_kernel,dropout,
                           layer_drop=layer_drop*(i+1)/num_layers)
            for i in range(num_layers)
        ])
        self.head=nn.Linear(model_dim,vocab_size)
    def forward(self, x, lengths):
        x, ol = self.sub(x, lengths); x = self.pos(x)
        B,T,_ = x.shape
        mask = torch.arange(T,device=x.device).unsqueeze(0) >= ol.unsqueeze(1)
        for blk in self.blocks: x = blk(x, mask)
        return F.log_softmax(self.head(x),dim=-1).permute(1,0,2), ol
    def count_params(self): return sum(p.numel() for p in self.parameters())

model = ConformerCTC(
    VOCAB_SIZE, N_MELS, MODEL_DIM, NUM_HEADS, NUM_LAYERS, FF_DIM,
    CONV_KERNEL, DROPOUT, LAYER_DROP
).to(DEVICE)
n = model.count_params()
print(f'Params: {n:,} ({n/1e6:.2f}M)'); assert n <= 5_000_000

swa_model = AveragedModel(model)

# ── Audio augmentation ────────────────────────────────────────────────────────
_mel_tf = T.MelSpectrogram(
    sample_rate=SAMPLE_RATE, n_fft=N_FFT, win_length=WIN_LENGTH,
    hop_length=HOP_LENGTH, n_mels=N_MELS, f_min=F_MIN, f_max=F_MAX, power=2.0)

def add_noise(wav, snr_range=(5, 20)):
    snr_db = random.uniform(*snr_range)
    sig_power   = wav.norm(p=2).clamp(min=1e-9)
    noise       = torch.randn_like(wav)
    noise_power = noise.norm(p=2).clamp(min=1e-9)
    scale = sig_power / (noise_power * (10 ** (snr_db / 20)))
    return (wav + scale * noise).clamp(-1, 1)

def add_reverb(wav, sr=SAMPLE_RATE):
    """Synthetic reverb: exponentially-decaying echo."""
    delay_ms = random.uniform(20, 80)
    decay    = random.uniform(0.15, 0.40)
    delay_samples = int(delay_ms * sr / 1000)
    reflected = F.pad(wav, (delay_samples, 0))[:, :wav.shape[1]]
    return (wav + decay * reflected).clamp(-1, 1)

_SPEED_RATES = [0.85, 0.9, 0.95, 1.05, 1.1, 1.15]

def load_wav(path):
    wav, sr = torchaudio.load(path)
    if wav.shape[0]>1: wav = wav.mean(0,keepdim=True)
    if sr != SAMPLE_RATE: wav = torchaudio.functional.resample(wav, sr, SAMPLE_RATE)
    return wav

def wav_to_mel(wav, max_len=MAX_MEL_LEN):
    mel = torch.log(_mel_tf(wav).squeeze(0).T.clamp(min=1e-9))
    return mel[:max_len]

def load_mel(path):
    return wav_to_mel(load_wav(path))

class NumbersDataset(Dataset):
    def __init__(self, csv_path, data_dir, training=False):
        self.df=pd.read_csv(csv_path); self.data_dir=data_dir; self.training=training
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        wav = load_wav(os.path.join(self.data_dir, row['filename']))
        if self.training:
            if random.random() < 0.6:
                rate = random.choice(_SPEED_RATES)
                wav  = torchaudio.functional.resample(
                    wav, int(SAMPLE_RATE*rate), SAMPLE_RATE)
            if random.random() < 0.5:
                wav = add_noise(wav, snr_range=(5, 20))
            if random.random() < 0.3:
                wav = add_reverb(wav)
        mel = wav_to_mel(wav)
        return mel, torch.tensor(
            encode(number_to_text(int(row['transcription']))), dtype=torch.long)

def collate(batch):
    mels,tgts=zip(*batch)
    ml=torch.tensor([m.size(0) for m in mels],dtype=torch.long)
    tl=torch.tensor([t.size(0) for t in tgts],dtype=torch.long)
    pad=torch.zeros(len(mels),ml.max(),N_MELS)
    for i,m in enumerate(mels): pad[i,:m.size(0)]=m
    return pad, ml, torch.cat(tgts), tl

def spec_augment(mels):
    mels=mels.clone(); B,T,F=mels.shape
    for _ in range(2):
        f=random.randint(0,FREQ_MASK_P); f0=random.randint(0,F-f); mels[:,:,f0:f0+f]=0
        t=random.randint(0,TIME_MASK_P); t0=random.randint(0,max(T-t,0)); mels[:,t0:t0+t,:]=0
    return mels

# ── Dataloaders ───────────────────────────────────────────────────────────────
train_ds  = NumbersDataset(os.path.join(DATA_DIR,'train.csv'), DATA_DIR, training=True)
dev_ds    = NumbersDataset(os.path.join(DATA_DIR,'dev.csv'),   DATA_DIR, training=False)
dev_df    = pd.read_csv(os.path.join(DATA_DIR,'dev.csv'))
train_spk = set(pd.read_csv(os.path.join(DATA_DIR,'train.csv'))['spk_id'].unique())

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, collate_fn=collate, pin_memory=True, drop_last=True)
dev_loader   = DataLoader(dev_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, collate_fn=collate, pin_memory=True)
print(f'Train: {len(train_ds)}  Dev: {len(dev_ds)}')

# ── CER / Evaluate ────────────────────────────────────────────────────────────
def cer(ref, hyp):
    if not ref: return 0. if not hyp else 1.
    r,h=list(ref),list(hyp)
    d=[[0]*(len(h)+1) for _ in range(len(r)+1)]
    for i in range(len(r)+1): d[i][0]=i
    for j in range(len(h)+1): d[0][j]=j
    for i in range(1,len(r)+1):
        for j in range(1,len(h)+1):
            d[i][j]=min(d[i-1][j]+1,d[i][j-1]+1,
                        d[i-1][j-1]+(0 if r[i-1]==h[j-1] else 1))
    return d[len(r)][len(h)]/len(r)

@torch.no_grad()
def evaluate(eval_model=None):
    m = eval_model if eval_model is not None else model
    m.eval()
    inD, ooD, si = [], [], 0
    for mels, ml, tgts, tl in dev_loader:
        lp, ol = m(mels.to(DEVICE), ml.to(DEVICE))
        hyps = ctc_greedy_decode(lp.cpu(), ol.cpu())
        off = 0
        for b in range(mels.size(0)):
            ref=decode(tgts[off:off+tl[b]].tolist()); off+=tl[b]
            v=cer(ref, decode(hyps[b]))
            (inD if dev_df.iloc[si].get('spk_id','') in train_spk else ooD).append(v)
            si+=1
    a=sum(inD)/len(inD) if inD else 0
    b=sum(ooD)/len(ooD) if ooD else 0
    model.train()
    return {'inD':a,'ooD':b,'score':2*a*b/(a+b)*100 if a+b>0 else 0}

# ── Optimizer / scheduler ─────────────────────────────────────────────────────
ctc_loss  = nn.CTCLoss(blank=BLANK_ID, reduction='mean', zero_infinity=True)
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
total_steps  = (len(train_loader) // ACCUM_STEPS) * MAX_EPOCHS
warmup_steps = min(WARMUP_STEPS, total_steps//10)

def lr_lambda(step):
    if step < warmup_steps: return step/max(1,warmup_steps)
    p=(step-warmup_steps)/max(1,total_steps-warmup_steps)
    return max(0., 0.5*(1.+math.cos(math.pi*p)))

scheduler = LambdaLR(optimizer, lr_lambda)
scaler    = torch.amp.GradScaler(enabled=USE_AMP)

# ── Training loop ─────────────────────────────────────────────────────────────
best_score      = float('inf')
best_model_path = f'{OUTPUT_DIR}/best_model.pt'
swa_active      = False

for epoch in range(1, MAX_EPOCHS+1):
    model.train()
    ep_loss = 0.
    optimizer.zero_grad()
    for i, (mels, ml, tgts, tl) in enumerate(
            tqdm(train_loader, desc=f'Ep {epoch}/{MAX_EPOCHS}', leave=False)):
        mels = spec_augment(mels).to(DEVICE)
        with torch.amp.autocast('cuda', enabled=USE_AMP):
            lp, ol = model(mels, ml.to(DEVICE))
            loss   = ctc_loss(lp, tgts.to(DEVICE), ol, tl.to(DEVICE)) / ACCUM_STEPS
        scaler.scale(loss).backward()
        ep_loss += loss.item() * ACCUM_STEPS
        if (i+1) % ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer); scaler.update()
            optimizer.zero_grad(); scheduler.step()

    if epoch >= SWA_START:
        swa_model.update_parameters(model)
        swa_active = True

    m = evaluate()
    print(f'Ep {epoch:3d} | loss={ep_loss/len(train_loader):.4f} | '
          f'CER inD={m["inD"]*100:.2f}% ooD={m["ooD"]*100:.2f}% '
          f'score={m["score"]:.2f}%'
          + (' [SWA]' if swa_active else ''))

    if m['score'] < best_score:
        best_score = m['score']
        torch.save(model.state_dict(), best_model_path)
        print(f'  -> Best! score={best_score:.2f}%')

print(f'\nTraining done. Best regular score: {best_score:.2f}%')

# ── SWA: update BN and compare ────────────────────────────────────────────────
if swa_active:
    print('Updating BatchNorm stats for SWA model...')
    swa_model.train()
    with torch.no_grad():
        for mels, ml, tgts, tl in tqdm(train_loader, desc='BN update', leave=False):
            swa_model(mels.to(DEVICE), ml.to(DEVICE))
    swa_metrics = evaluate(eval_model=swa_model)
    print(f'SWA: inD={swa_metrics["inD"]*100:.2f}% ooD={swa_metrics["ooD"]*100:.2f}% score={swa_metrics["score"]:.2f}%')
    if swa_metrics['score'] < best_score:
        print('SWA model is better! Using SWA for inference.')
        swa_state = {k.replace('module.', ''): v
                     for k, v in swa_model.state_dict().items()
                     if 'n_averaged' not in k}
        torch.save(swa_state, best_model_path)
    else:
        print('Regular model is better. Keeping best checkpoint.')

# ── Inference: Greedy + TTA (3 speeds, majority vote) ────────────────────────
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
model.eval()

test_df = pd.read_csv(os.path.join(DATA_DIR,'test.csv'))
rows = []

with torch.no_grad():
    for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc='Inference+TTA'):
        wav_orig = load_wav(os.path.join(DATA_DIR, row['filename']))
        preds = []
        for speed in TTA_SPEEDS:
            if speed != 1.0:
                wav = torchaudio.functional.resample(
                    wav_orig, int(SAMPLE_RATE * speed), SAMPLE_RATE)
            else:
                wav = wav_orig
            mel = wav_to_mel(wav).unsqueeze(0).to(DEVICE)
            lp, ol = model(mel, torch.tensor([mel.size(1)], device=DEVICE))
            token_ids = ctc_greedy_decode(lp.cpu(), ol.cpu())[0]  # greedy (fast & robust)
            preds.append(safe_parse(decode(token_ids)))

        # Majority vote; if all 3 disagree, use original speed result
        num = Counter(preds).most_common(1)[0][0]
        rows.append({'filename': row['filename'], 'transcription': num})

sub = pd.DataFrame(rows)
sub.to_csv(f'{OUTPUT_DIR}/submission.csv', index=False)

n_fallback = (sub['transcription']==1_000).sum()
n_below    = (sub['transcription']<1_000).sum()
print(f'\nTotal: {len(sub)} | Fallback(=1000): {n_fallback} ({n_fallback/len(sub)*100:.1f}%)')
print(f'Below 1000: {n_below}')
print(sub['transcription'].describe())
print(sub.head())